In [3]:
from pathlib import Path
import pandas as pd, numpy as np, json, matplotlib.pyplot as plt, seaborn as sns
try:
    from wordcloud import WordCloud
    HAS_WC = True
    print("wordcloud available")
except ImportError:
    HAS_WC = False
    print("wordcloud not installed — Fig3 will be bar chart fallback. pip install wordcloud")

BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
RES = BASE / "results"
FIG = BASE / "figures"
FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


wordcloud available


In [4]:
summary = pd.read_csv(RES / "flagging_rate_summary.csv")
s07 = summary[summary['threshold']==0.7]
plt.figure(figsize=(8,5))
colors = {"baseline":"#4e79a7","crisis":"#e15759","neutral_generic":"#59a14f","neutral_injected":"#f28e2b"}
plt.bar(s07['split'], s07['FR'], color=[colors.get(x,"grey") for x in s07['split']])
plt.ylabel("Flagging Rate (threshold 0.7)")
plt.title("Flagging Rate: Baseline vs Crisis vs Neutral Probes")
plt.xticks(rotation=15)
for i, v in enumerate(s07['FR']):
    plt.text(i, v+0.01, f"{v:.1%}\n(n={int(s07.iloc[i]['n'])})", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG / "fig1_flagging_rate.png", dpi=300)
plt.close()
print("Fig1 -> figures/fig1_flagging_rate.png")


Fig1 -> figures/fig1_flagging_rate.png


In [ ]:
neutral = pd.read_csv(RES / "neutral_scored.csv")
plt.figure(figsize=(8,5))
sns.histplot(data=neutral, x="toxicity", hue="type", bins=15, kde=True, palette=["#59a14f","#f28e2b"], alpha=0.6)
plt.axvline(0.7, color="red", linestyle="--", label="flag threshold 0.7")
plt.title("Toxicity Distribution: Generic vs Crisis-Injected Neutral Statements")
plt.xlabel("Toxicity Score")
plt.legend()
plt.tight_layout()
plt.savefig(FIG / "fig2_neutral_distribution.png", dpi=300)
plt.close()
print("Fig2 -> figures/fig2_neutral_distribution.png")

Fig2 -> figures/fig2_neutral_distribution.png


In [9]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd
import seaborn as sns

neutral = pd.read_csv(RES / "neutral_scored.csv")
plt.figure(figsize=(9, 5.5))

palette = {
    "generic": "#59a14f",   # Green
    "injected": "#f28e2b"   # Orange
}

# 1. Plot the histogram (disable Seaborn's default legend to prevent conflicts)
ax = sns.histplot(
    data=neutral, 
    x="toxicity", 
    hue="type", 
    bins=15, 
    kde=True, 
    palette=palette, 
    alpha=0.6,
    legend=False
)

# 2. Add the vertical threshold line
plt.axvline(0.7, color="red", linestyle="--", linewidth=2)

# 3. Define explicit descriptive labels and handles
legend_elements = [
    mpatches.Patch(
        facecolor="#59a14f", 
        edgecolor="#59a14f", 
        alpha=0.6, 
        label="Generic Neutral Statements"
    ),
    mpatches.Patch(
        facecolor="#f28e2b", 
        edgecolor="#f28e2b", 
        alpha=0.6, 
        label="Crisis-Injected Neutral Statements"
    ),
    Line2D(
        [0], [0], 
        color="red", 
        linestyle="--", 
        linewidth=2, 
        label="Flag Threshold (0.7)"
    )
]

# 4. Render the legend with full descriptions
plt.legend(
    handles=legend_elements, 
    title="Legend / Descriptions", 
    title_fontsize=11,
    fontsize=10, 
    loc="upper right",
    frameon=True
)

plt.title("Toxicity Distribution: Generic vs Crisis-Injected Neutral Statements", fontsize=13, pad=12)
plt.xlabel("Toxicity Score", fontsize=11)
plt.ylabel("Count / Frequency", fontsize=11)

plt.tight_layout()
plt.savefig(FIG / "fig2_neutral_distribution_v2.png", dpi=300)
plt.close()
print("Fig2 -> figures/fig2_neutral_distribution_v2.png")

Fig2 -> figures/fig2_neutral_distribution_v2.png


In [11]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

if (RES / "crisis_scored.csv").exists():
    crisis = pd.read_csv(RES / "crisis_scored.csv")
    baseline = pd.read_csv(RES / "baseline_scored.csv")

    plt.figure(figsize=(8.5, 5.5))

    # Combine datasets
    df2 = pd.concat([
        crisis.assign(split="crisis")[["toxicity", "split"]],
        baseline.assign(split="baseline")[["toxicity", "split"]]
    ])

    # Explicit palette mapping
    palette = {
        "crisis": "#e15759",    # Red
        "baseline": "#4e79a7"   # Blue
    }

    # 1. Plot the violin plot
    ax = sns.violinplot(
        data=df2, 
        x="split", 
        y="toxicity", 
        order=["crisis", "baseline"],
        palette=palette,
        inner="quartile",      # Shows median and quartiles inside the violins
        cut=0                  # Constrains violin to the observed data range
    )

    # 2. Rename x-axis tick labels for clarity
    ax.set_xticklabels(["Crisis Tweets", "Baseline Tweets"], fontsize=11)

    # 3. Add custom legend with full descriptions
    legend_elements = [
        mpatches.Patch(
            facecolor="#e15759", 
            edgecolor="#e15759", 
            alpha=0.8, 
            label="Crisis Tweets (Target Event)"
        ),
        mpatches.Patch(
            facecolor="#4e79a7", 
            edgecolor="#4e79a7", 
            alpha=0.8, 
            label="Baseline Tweets (Control/General)"
        )
    ]

    plt.legend(
        handles=legend_elements, 
        title="Dataset Split", 
        title_fontsize=11,
        fontsize=10, 
        loc="upper right", 
        frameon=True
    )

    # 4. Labels and Title
    plt.title("Toxicity Distribution: Crisis vs Baseline Tweets", fontsize=13, pad=12)
    plt.xlabel("Dataset Category", fontsize=11)
    plt.ylabel("Toxicity Score", fontsize=11)

    plt.tight_layout()
    plt.savefig(FIG / "fig2b_crisis_baseline_violin.png", dpi=300)
    plt.close()
    print("Fig2b -> figures/fig2b_crisis_baseline_violin.png")

C:\Users\phoen\AppData\Local\Temp\ipykernel_17036\3067909227.py:25: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
C:\Users\phoen\AppData\Local\Temp\ipykernel_17036\3067909227.py:36: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(["Crisis Tweets", "Baseline Tweets"], fontsize=11)


Fig2b -> figures/fig2b_crisis_baseline_violin.png


In [12]:
hotspots = pd.read_csv(RES / "top20_hotspots.csv")
if HAS_WC:
    text = " ".join([ (w+" ")*int(max(1, d*1000)) for w, d in zip(hotspots['word'], hotspots['delta_tfidf']) ])
    wc = WordCloud(width=800, height=400, background_color="white", colormap="Reds").generate(text)
    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("High-Risk Crisis Keywords (TF-IDF delta weighted)")
    plt.tight_layout()
    plt.savefig(FIG / "fig3_hotspot_wordcloud.png", dpi=300)
    plt.close()
else:
    plt.figure(figsize=(10,5))
    plt.barh(hotspots['word'].head(15)[::-1], hotspots['delta_tfidf'].head(15)[::-1], color="#e15759")
    plt.title("High-Risk Crisis Keywords (TF-IDF delta)")
    plt.tight_layout()
    plt.savefig(FIG / "fig3_hotspot_wordcloud.png", dpi=300)
    plt.close()
print("Fig3 -> figures/fig3_hotspot_wordcloud.png")


Fig3 -> figures/fig3_hotspot_wordcloud.png


In [13]:
if (RES / "decision_boundary_weights.csv").exists():
    w = pd.read_csv(RES / "decision_boundary_weights.csv")
    plt.figure(figsize=(10,6))
    w_sorted = w.sort_values("weight")
    colors2 = ["#4e79a7" if x=="pro-true" else "#e15759" for x in w_sorted['direction']]
    plt.barh(w_sorted['feature'], w_sorted['weight'], color=colors2)
    plt.title("Decision Boundary: Top Pro-Misinfo vs Pro-True Features (Baseline-trained LR)")
    plt.xlabel("LR weight")
    plt.tight_layout()
    plt.savefig(FIG / "fig4_decision_boundary.png", dpi=300)
    plt.close()
    print("Fig4 -> figures/fig4_decision_boundary.png")


Fig4 -> figures/fig4_decision_boundary.png


In [14]:
s07_dict = s07.set_index('split')['FR'].to_dict()
snippet = f"""
## Quick Numbers for Paper (auto-generated)

- Baseline FR@0.7: {s07_dict.get('baseline',0):.1%}
- Crisis FR@0.7: {s07_dict.get('crisis',0):.1%} (delta {(s07_dict.get('crisis',0)-s07_dict.get('baseline',0)):+.1%})
- Neutral generic FR@0.7: {s07_dict.get('neutral_generic',0):.1%}
- Neutral injected FR@0.7: {s07_dict.get('neutral_injected',0):.1%} (delta {(s07_dict.get('neutral_injected',0)-s07_dict.get('neutral_generic',0)):+.1%})
- Hotspots: {', '.join(hotspots['word'].head(10))}
"""
(Path(BASE/"to-do/logs/day06_numbers.md")).write_text(snippet, encoding="utf-8")
print(snippet)
print("All figures done. Log -> to-do/logs/day06_numbers.md")



## Quick Numbers for Paper (auto-generated)

- Baseline FR@0.7: 5.4%
- Crisis FR@0.7: 74.6% (delta +69.2%)
- Neutral generic FR@0.7: 0.0%
- Neutral injected FR@0.7: 0.0% (delta +0.0%)
- Hotspots: ukraine, russian, putin, war, russia, embassy, canada, tank, russian embassy, embassy canada

All figures done. Log -> to-do/logs/day06_numbers.md
